---
# Lab 2

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Lab1").getOrCreate()

## Section 1 — Practice Exercises

In [2]:
employees = spark.createDataFrame([
    (1, "Anna",  "Engineering", 95000),
    (2, "Ben",   "Marketing",   72000),
    (3, "Clara", "Engineering", 88000),
    (4, "Dan",   "HR",          65000),
    (5, "Eva",   "Marketing",   79000),
    (6, "Frank", "Engineering", 91000),
], ["id", "name", "dept", "salary"])
employees.show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1| Anna|Engineering| 95000|
|  2|  Ben|  Marketing| 72000|
|  3|Clara|Engineering| 88000|
|  4|  Dan|         HR| 65000|
|  5|  Eva|  Marketing| 79000|
|  6|Frank|Engineering| 91000|
+---+-----+-----------+------+



### Exercise 1 — Engineering employees with salary > 90,000

In [6]:
from pyspark.sql.functions import col

employees.filter(
    (col("dept") == "Engineering") &
    (col("salary") > 90000)
).show()

+---+-----+-----------+------+
| id| name|       dept|salary|
+---+-----+-----------+------+
|  1| Anna|Engineering| 95000|
|  6|Frank|Engineering| 91000|
+---+-----+-----------+------+



### Exercise 2 — salary_grade column (High / Standard)

In [7]:
result = employees.withColumn(
    "salary_grade",
    when(col("salary") >= 80000, "High").otherwise("Standard")
)
result.show()

+---+-----+-----------+------+------------+
| id| name|       dept|salary|salary_grade|
+---+-----+-----------+------+------------+
|  1| Anna|Engineering| 95000|        High|
|  2|  Ben|  Marketing| 72000|    Standard|
|  3|Clara|Engineering| 88000|        High|
|  4|  Dan|         HR| 65000|    Standard|
|  5|  Eva|  Marketing| 79000|    Standard|
|  6|Frank|Engineering| 91000|        High|
+---+-----+-----------+------+------------+



### Exercise 3 — Average salary per department (sorted descending)

In [8]:
result = (employees
    .groupBy("dept")
    .agg(avg("salary").alias("avg_salary"))
    .orderBy("avg_salary", ascending=False)
)
result.show()

+-----------+-----------------+
|       dept|       avg_salary|
+-----------+-----------------+
|Engineering|91333.33333333333|
|  Marketing|          75500.0|
|         HR|          65000.0|
+-----------+-----------------+



### Exercise 4 — Count distinct departments

In [10]:
from pyspark.sql.functions import countDistinct
result = employees.select(countDistinct("dept").alias("distinct_depts"))
result.show()

+--------------+
|distinct_depts|
+--------------+
|             3|
+--------------+



### Exercise 5 — Rename `dept` → `department`, drop `id`

In [11]:
result = employees.withColumnRenamed("dept", "department").drop("id")
result.show()

+-----+-----------+------+
| name| department|salary|
+-----+-----------+------+
| Anna|Engineering| 95000|
|  Ben|  Marketing| 72000|
|Clara|Engineering| 88000|
|  Dan|         HR| 65000|
|  Eva|  Marketing| 79000|
|Frank|Engineering| 91000|
+-----+-----------+------+



## Section 2 — Window Function Exercises

### Exercise 1 — Running total of salary ordered by id

In [13]:
w = Window.orderBy("id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = employees.withColumn("running_salary_total", sum("salary").over(w))
result.show()

+---+-----+-----------+------+--------------------+
| id| name|       dept|salary|running_salary_total|
+---+-----+-----------+------+--------------------+
|  1| Anna|Engineering| 95000|               95000|
|  2|  Ben|  Marketing| 72000|              167000|
|  3|Clara|Engineering| 88000|              255000|
|  4|  Dan|         HR| 65000|              320000|
|  5|  Eva|  Marketing| 79000|              399000|
|  6|Frank|Engineering| 91000|              490000|
+---+-----+-----------+------+--------------------+



### Exercise 2 — above_avg / below_avg per department

In [15]:
w_dept = Window.partitionBy("dept")
result = (employees
    .withColumn("dept_avg", avg("salary").over(w_dept))
    .withColumn("vs_dept_avg",
        when(col("salary") >= col("dept_avg"), "above_avg")
         .otherwise("below_avg"))
    .drop("dept_avg")
)
result.show()

+---+-----+-----------+------+-----------+
| id| name|       dept|salary|vs_dept_avg|
+---+-----+-----------+------+-----------+
|  1| Anna|Engineering| 95000|  above_avg|
|  3|Clara|Engineering| 88000|  below_avg|
|  6|Frank|Engineering| 91000|  below_avg|
|  4|  Dan|         HR| 65000|  above_avg|
|  2|  Ben|  Marketing| 72000|  below_avg|
|  5|  Eva|  Marketing| 79000|  above_avg|
+---+-----+-----------+------+-----------+



### Exercise 3 — Top 3 Engineering employees by salary (window function)

In [16]:
w_eng = Window.partitionBy("dept").orderBy(col("salary").desc())
result = (employees
    .withColumn("rank", rank().over(w_eng))
    .filter((col("dept") == "Engineering") & (col("rank") <= 3))
)
result.show()

+---+-----+-----------+------+----+
| id| name|       dept|salary|rank|
+---+-----+-----------+------+----+
|  1| Anna|Engineering| 95000|   1|
|  6|Frank|Engineering| 91000|   2|
|  3|Clara|Engineering| 88000|   3|
+---+-----+-----------+------+----+



## Section 3 — String Function Exercises

In [17]:
logs = spark.createDataFrame([
    ("2025-06-01 08:15:32 ERROR disk full",),
    ("2025-06-01 09:00:00 INFO  service started",),
    ("2025-06-02 14:45:11 WARN  memory low",),
    ("2025-06-03 23:59:59 ERROR connection timeout",),
], ["log_entry"])

emails_df = spark.createDataFrame([
    ("alice@gmail.com",),
    ("bob.smith@company.org",),
    ("clara99@yahoo.com",),
], ["email"])

names_df = spark.createDataFrame([
    ("  john doe 1st  ",),
    (" JANE  SMITH2  ",),
    ("  ali HASSAN  ",),
], ["name"])

### Exercise 1 — Extract date and log level from log entries

In [18]:
result = (logs
    .withColumn("log_date",  regexp_extract("log_entry", r'^(\d{4}-\d{2}-\d{2})', 1))
    .withColumn("log_level", trim(regexp_extract("log_entry", r'\d{2}:\d{2}:\d{2}\s+(\S+)', 1)))
)
result.show(truncate=False)

+--------------------------------------------+----------+---------+
|log_entry                                   |log_date  |log_level|
+--------------------------------------------+----------+---------+
|2025-06-01 08:15:32 ERROR disk full         |2025-06-01|ERROR    |
|2025-06-01 09:00:00 INFO  service started   |2025-06-01|INFO     |
|2025-06-02 14:45:11 WARN  memory low        |2025-06-02|WARN     |
|2025-06-03 23:59:59 ERROR connection timeout|2025-06-03|ERROR    |
+--------------------------------------------+----------+---------+



### Exercise 2 — Extract username and domain from email

In [19]:
result = (emails_df
    .withColumn("username", regexp_extract("email", r'^([^@]+)@', 1))
    .withColumn("domain",   regexp_extract("email", r'@(.+)$', 1))
)
result.show(truncate=False)

+---------------------+---------+-----------+
|email                |username |domain     |
+---------------------+---------+-----------+
|alice@gmail.com      |alice    |gmail.com  |
|bob.smith@company.org|bob.smith|company.org|
|clara99@yahoo.com    |clara99  |yahoo.com  |
+---------------------+---------+-----------+



### Exercise 3 — Clean name: trim, proper case, remove digits

In [20]:
result = (names_df
    .withColumn("name",trim(col("name")))
    .withColumn("name", initcap(col("name")))
    .withColumn("name", regexp_replace("name", r'[0-9]', ""))
    .withColumn("name", trim(col("name")))
)
result.show(truncate=False)

+-----------+
|name       |
+-----------+
|John Doe st|
|Jane  Smith|
|Ali Hassan |
+-----------+



## Section 4 — Date & Time Exercises

In [21]:
orders = spark.createDataFrame([
    (1, "2025-01-05"),
    (2, "2025-04-12"),
    (3, "2025-06-21"),
    (4, "2025-09-07"),
    (5, "2025-11-29"),
], ["order_id", "order_date"])
orders = orders.withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))
orders.show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2025-01-05|
|       2|2025-04-12|
|       3|2025-06-21|
|       4|2025-09-07|
|       5|2025-11-29|
+--------+----------+



### Exercise 1 — Order age in days, quarter, last day of month

In [23]:
result = (orders
    .withColumn("age_days",   datediff(current_date(), "order_date"))
    .withColumn("quarter",
        when(month("order_date").between(1, 3),  "Q1")
         .when(month("order_date").between(4, 6),  "Q2")
         .when(month("order_date").between(7, 9),  "Q3")
         .otherwise("Q4"))
    .withColumn("last_day_of_month", last_day("order_date"))
)
result.show()

+--------+----------+--------+-------+-----------------+
|order_id|order_date|age_days|quarter|last_day_of_month|
+--------+----------+--------+-------+-----------------+
|       1|2025-01-05|     514|     Q1|       2025-01-31|
|       2|2025-04-12|     417|     Q2|       2025-04-30|
|       3|2025-06-21|     347|     Q2|       2025-06-30|
|       4|2025-09-07|     269|     Q3|       2025-09-30|
|       5|2025-11-29|     186|     Q4|       2025-11-30|
+--------+----------+--------+-------+-----------------+



### Exercise 2 — Orders placed on a weekend (Saturday or Sunday)

In [24]:
# dayofweek: 1=Sunday, 7=Saturday
result = orders.filter(dayofweek("order_date").isin(1, 7))
result.show()

+--------+----------+
|order_id|order_date|
+--------+----------+
|       1|2025-01-05|
|       2|2025-04-12|
|       3|2025-06-21|
|       4|2025-09-07|
|       5|2025-11-29|
+--------+----------+



## Section 5 — Array Function Exercises

In [25]:
products = spark.createDataFrame([
    (1, "Laptop",  ["electronics", "computers", "work"]),
    (2, "T-Shirt", ["clothing", "fashion"]),
    (3, "Phone",   ["electronics", "mobile"]),
    (4, "Headset", ["electronics", "audio", "computers"]),
    (5, "Jacket",  ["clothing", "fashion", "sale"]),
], ["id", "product", "tags"])

promo_products = spark.createDataFrame([
    (1, ["sale", "featured"]),
    (2, ["new",  "sale"]),
    (3, ["featured"]),
    (4, ["sale", "electronics"]),
    (5, ["clearance"]),
], ["id", "promo_tags"])

### Exercise 1 — Products with more than 2 tags

In [26]:
result = products.filter(size("tags") > 2)
result.show(truncate=False)

+---+-------+-------------------------------+
|id |product|tags                           |
+---+-------+-------------------------------+
|1  |Laptop |[electronics, computers, work] |
|4  |Headset|[electronics, audio, computers]|
|5  |Jacket |[clothing, fashion, sale]      |
+---+-------+-------------------------------+



### Exercise 2 — has_sale_tag column from promo_tags

In [28]:
result = promo_products.withColumn("has_sale_tag", array_contains("promo_tags", "sale"))
result.show(truncate=False)

+---+-------------------+------------+
|id |promo_tags         |has_sale_tag|
+---+-------------------+------------+
|1  |[sale, featured]   |true        |
|2  |[new, sale]        |true        |
|3  |[featured]         |false       |
|4  |[sale, electronics]|true        |
|5  |[clearance]        |false       |
+---+-------------------+------------+



### Exercise 3 — Count products per tag using explode() + groupBy()

In [29]:
result = (products
    .withColumn("tag", explode("tags"))
    .groupBy("tag")
    .agg(count("id").alias("product_count"))
    .orderBy("product_count", ascending=False)
)
result.show()

+-----------+-------------+
|        tag|product_count|
+-----------+-------------+
|electronics|            3|
|  computers|            2|
|   clothing|            2|
|    fashion|            2|
|       work|            1|
|     mobile|            1|
|      audio|            1|
|       sale|            1|
+-----------+-------------+

